# Question Answering Pipeline

**Authors:** Roberto Herrera and Bach Nguyen  
**Datasets:** HotPotQA and QASPER

We implement a retrieval-augmented extractive question answering pipeline. Stage 1 selects relevant text through chunking, embedding, and ChromaDB retrieval, implementing the assignment’s summarization stage. Stage 2 uses an extractive QA model to predict an answer from the retrieved context.

## 1. Setup

We define dataset locations, load the pipeline dependencies, and create a results directory. Experiment settings and outputs are saved with each run.

In [2]:
import os
import re
import ast
import json
import random
from datetime import datetime
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFaceEmbeddings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOTPOTQA_PATH = Path(
    "/root/.cache/kagglehub/datasets/"
    "jeromeblanchet/hotpotqa-question-answering-dataset/"
    "versions/1/hotpot_train_v1.1.json"
)
QASPER_DIR = Path(
    "/root/.cache/kagglehub/datasets/"
    "thedevastator/qasper-nlp-questions-and-evidence/versions/2"
)

# change when running on different machine
MACHINE_LABEL = "laptopRH"
RESULTS_DIR = Path("results") / MACHINE_LABEL
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 2. Load datasets
HotPotQA provides candidate Wikipedia paragraphs for each question.
QASPER groups questions and answer annotations by research paper.

In [4]:
def parse_cell(s):
    s = re.sub(r",\s*dtype=\w+\)", ")", s)
    s = s.replace("array(", "(")
    return ast.literal_eval(s)


def paper_paragraphs(row):
    full_text = parse_cell(row["full_text"])
    return [
        (section, paragraph)
        for section, paragraphs in zip(
            full_text["section_name"], full_text["paragraphs"]
        )
        for paragraph in paragraphs
        if paragraph.strip()
    ]


with HOTPOTQA_PATH.open(encoding="utf-8") as f:
    hotpot = json.load(f)

qasper = pd.read_csv(QASPER_DIR / "train.csv", dtype={"id": str})

print("HotPotQA questions:", len(hotpot))
print("QASPER papers:", len(qasper))

HotPotQA questions: 90447
QASPER papers: 888


## 3. Chunking

We compare recursive and semantic chunking. Recursive splitting uses a token-counted size cap, while semantic splitting identifies boundaries using a percentile threshold on adjacent text-unit embedding distances. Oversized sentences and semantic groups are split to respect the configured cap. Both methods use the active embedding tokenizer and preserve source metadata and character offsets.

In [7]:
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")


def make_splitter(chunk_size, chunk_overlap, tokenizer, token_limit):
    if not 0 <= chunk_overlap < chunk_size <= token_limit:
        raise ValueError(
            "Require 0 <= overlap < chunk_size <= embedding token limit."
        )

    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=lambda text: len(tokenizer.encode(
            text, add_special_tokens=True, truncation=False
        )),
        add_start_index=True,
    )


def semantic_chunks(text, metadata, embedder, splitter, percentile):
    if not 0 < percentile < 100:
        raise ValueError("percentile must be between 0 and 100")

    units = []
    for sentence in nlp(text).sents:
        if not sentence.text.strip():
            continue
        for piece in splitter.create_documents([sentence.text]):
            start = sentence.start_char + piece.metadata["start_index"]
            units.append((piece.page_content, start))

    if not units:
        return []

    breaks = []
    if len(units) > 1:
        vectors = np.asarray(
            embedder.embed_documents([text for text, _ in units]),
            dtype=float,
        )
        vectors /= np.maximum(
            np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12
        )
        distances = 1 - np.clip(
            np.sum(vectors[:-1] * vectors[1:], axis=1), -1, 1
        )
        threshold = np.percentile(distances, percentile)
        breaks = [
            i + 1 for i, distance in enumerate(distances)
            if distance > threshold
        ]

    docs = []
    first = 0
    for last in breaks + [len(units)]:
        start = units[first][1]
        end = units[last - 1][1] + len(units[last - 1][0])
        pieces = splitter.create_documents(
            [text[start:end]], metadatas=[metadata]
        )
        for piece in pieces:
            piece.metadata["start_index"] += start
            docs.append(piece)
        first = last

    return docs

## 4. Store chunks in ChromaDB

Each chunk is embedded and stored with its source identifier and character range.

HotPotQA uses a separate collection for each question’s candidate paragraphs. QASPER reuses a paper’s collection across its questions. Collections use cosine distance to compare question and chunk embeddings.

In [12]:
from uuid import uuid4
from langchain_chroma import Chroma


def hotpot_paragraphs(example):
    return [
        (" ".join(sentences), {"title": title, "para_id": pid})
        for pid, (title, sentences) in enumerate(example["context"])
    ]


def qasper_paragraphs(row):
    return [
        (paragraph, {"section": section, "para_id": pid})
        for pid, (section, paragraph) in enumerate(paper_paragraphs(row))
    ]


def build_store(paragraphs, embedder, chunking, chunk_size,
                chunk_overlap=0, percentile=None, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    started = now()

    if chunking not in {"recursive", "semantic"}:
        raise ValueError("chunking must be 'recursive' or 'semantic'")
    if chunking == "semantic":
        if percentile is None or not 0 < percentile < 100:
            raise ValueError("Semantic chunking requires a percentile in (0, 100).")
        if chunk_overlap != 0:
            raise ValueError("Semantic chunking currently requires zero overlap.")

    tokenizer = embedder._client.tokenizer
    splitter = make_splitter(
        chunk_size, chunk_overlap, tokenizer,
        embedder._client.max_seq_length,
    )

    docs = []
    for text, metadata in paragraphs:
        if chunking == "recursive":
            pieces = splitter.create_documents(
                [text], metadatas=[metadata]
            )
        else:
            pieces = semantic_chunks(
                text, metadata, embedder, splitter, percentile
            )

        for piece in pieces:
            start = piece.metadata["start_index"]
            end = start + len(piece.page_content)
            if start < 0 or text[start:end] != piece.page_content:
                raise ValueError("Chunk offsets do not match the source text.")

            token_count = len(tokenizer.encode(
                piece.page_content, add_special_tokens=True,
                truncation=False,
            ))
            if token_count > chunk_size:
                raise ValueError("A chunk exceeds its configured token cap.")

            piece.metadata.update({
                "end_index": end,
                "chunk_id": f"{metadata['para_id']}:{start}:{end}",
                "embedding_tokens": token_count,
            })
            docs.append(piece)

    chunked = now()
    if timings is not None:
        timings["chunking_s"] = chunked - started
        timings["embedding_store_s"] = 0.0

    store = None
    if docs:
        store = Chroma.from_documents(
            documents=docs,
            embedding=embedder,
            ids=[doc.metadata["chunk_id"] for doc in docs],
            collection_name="retrieval_" + uuid4().hex,
            collection_metadata={"hnsw:space": "cosine"},
        )

    finished = now()
    if timings is not None:
        if docs:
            timings["embedding_store_s"] = finished - chunked
        timings["build_total_s"] = finished - started

    return store

## 5. Retrieve relevant chunks

Single-pass retrieval searches using the original question. Two-hop retrieval uses the first retrieved chunk to expand the question, then searches again for additional chunks. Both return up to k unique chunks. We retain cosine distances and search-stage information to examine whether the second search adds useful evidence. Distances from different searches are not treated as one combined ranking.

In [13]:
def retrieve(store, question, k, method, query_tokenizer,
             max_query_tokens, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    started = now()

    if k < 1:
        raise ValueError("k must be at least 1")
    if method not in {"single_pass", "two_hop"}:
        raise ValueError("Unknown retrieval method.")
    if store is None:
        if timings is not None:
            timings["retrieval_s"] = now() - started
        return {"chunks": [], "expanded_query": None}

    def token_count(text):
        return len(query_tokenizer.encode(text, add_special_tokens=True))

    if token_count(question) > max_query_tokens:
        raise ValueError("Question exceeds the embedding input limit.")

    first = store.similarity_search_with_score(
        question, k=k if method == "single_pass" else 1
    )
    selected = [(doc, distance, 1) for doc, distance in first]
    expanded_query = None

    if method == "two_hop" and k > 1 and first:
        evidence = first[0][0].page_content
        offsets = query_tokenizer(
            evidence,
            add_special_tokens=False,
            return_offsets_mapping=True,
            truncation=False,
        )["offset_mapping"]

        end = min(len(offsets), max_query_tokens - token_count(question))
        while end > 0:
            candidate = question + "\n" + evidence[:offsets[end - 1][1]]
            if token_count(candidate) <= max_query_tokens:
                expanded_query = candidate
                break
            end -= 1

        if expanded_query is None:
            raise ValueError("No first-hop evidence fits beside the question.")

        seen = {doc.metadata["chunk_id"] for doc, _, _ in selected}
        second = store.similarity_search_with_score(expanded_query, k=k)

        for doc, distance in second:
            chunk_id = doc.metadata["chunk_id"]
            if chunk_id not in seen:
                selected.append((doc, distance, 2))
                seen.add(chunk_id)
            if len(selected) == k:
                break

    result = {
        "chunks": [
            {
                "rank": rank,
                "text": doc.page_content,
                "metadata": dict(doc.metadata),
                "distance": float(distance),
                "hop": hop,
            }
            for rank, (doc, distance, hop) in enumerate(selected, start=1)
        ],
        "expanded_query": expanded_query,
    }

    if timings is not None:
        timings["retrieval_s"] = now() - started

    return result

## 6. Prepare the QA context

Retrieved chunks are added in retrieval order within the reader’s input budget, including the question and special tokens. If a chunk does not fit, only its fitting prefix is included and packing stops. We record the final context, token count, and retained source ranges to distinguish retrieval failures from evidence lost during packing.

In [14]:
def pack_context(question, chunks, qa_tokenizer,
                 max_tokens=512, timings=None):
    started = perf_counter()

    def input_length(context):
        return len(qa_tokenizer(
            question, context, add_special_tokens=True,
            truncation=False,
        )["input_ids"])

    if input_length("") >= max_tokens:
        raise ValueError("The question leaves no room for QA context.")

    context = ""
    included = []

    for chunk in chunks:
        text = chunk["text"]
        separator = "\n\n" if context else ""
        candidate = context + separator + text
        truncated = False

        if input_length(candidate) > max_tokens:
            offsets = qa_tokenizer(
                text, add_special_tokens=False,
                return_offsets_mapping=True, truncation=False,
            )["offset_mapping"]
            end = min(len(offsets), max_tokens)

            while end > 0:
                prefix = text[:offsets[end - 1][1]]
                candidate = context + separator + prefix
                if input_length(candidate) <= max_tokens:
                    text = prefix
                    break
                end -= 1

            if end == 0:
                break
            truncated = True

        context_start = len(context) + len(separator)
        context = candidate
        metadata = dict(chunk["metadata"])
        metadata["end_index"] = metadata["start_index"] + len(text)

        included.append({
            "rank": chunk["rank"],
            "text": text,
            "metadata": metadata,
            "context_start": context_start,
            "context_end": len(context),
            "truncated": truncated,
        })

        if truncated:
            break

    result = {
        "context": context,
        "input_tokens": input_length(context),
        "included_chunks": included,
    }

    if timings is not None:
        timings["packing_s"] = perf_counter() - started

    return result